In [1]:
import torch
print(f"Ekran kartı devrede mi?: {torch.cuda.is_available()}")

Ekran kartı devrede mi?: True


In [2]:
import os
from huggingface_hub import login

# Token: ortam degiskeni HF_TOKEN veya huggingface-cli login
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("✅ HuggingFace'e giriş yapıldı!")
else:
    print("⚠️ HF_TOKEN tanımlı değil; public modeller için giriş atlanabilir.")

✅ HuggingFace'e giriş yapıldı!


In [3]:
from datasets import load_dataset, Dataset

print("⏳ Sentetik veri seti indiriliyor...")
# Hızlı eğitim ve deneme için ilk 3000 veriyi alıyoruz.
dataset = load_dataset("starmpcc/Asclepius-Synthetic-Clinical-Notes", split="train[:3000]")

training_data = []
for item in dataset:
    # Modelin belleğini taşırmamak için not uzunluğunu sınırlandırıyoruz
    note = item['note'][:3000] 
    
    # Modelin doktor rolüne girmesi için instruction şablonu
    instruction = f"""You are a medical doctor. Write a detailed clinical discharge note (epikriz) for the following patient.

Patient Information:
- Patient ID: {item.get('patient_id', 'Unknown')}
- Task: Discharge Summary generation based on clinical context.

Write a structured clinical note with visit details, diagnosis, treatment, and discharge plan."""

    # Qwen / LLaMA formatına uygun birleştirme
    formatted_text = f"### Instruction:\n{instruction}\n\n### Response:\n{note}"
    
    training_data.append({"text": formatted_text})

hf_dataset = Dataset.from_list(training_data)

print(f"✅ {len(hf_dataset)} sentetik eğitim örneği hazırlandı!")

⏳ Sentetik veri seti indiriliyor...
✅ 3000 sentetik eğitim örneği hazırlandı!


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_name = "Qwen/Qwen2.5-3B-Instruct"

# 4-bit Quantization ayarları (Ekran kartını rahatlatır)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("⏳ Qwen-3B modeli ve Tokenizer indiriliyor...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token 

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto" 
)

# LoRA Ayarları (Sadece belirli ağırlıkları eğiterek bellek tasarrufu sağlar)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ Model hazır! Eğitilecek parametre oranı: %{100*trainable/total:.2f}")

⏳ Qwen-3B modeli ve Tokenizer indiriliyor...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ Model hazır! Eğitilecek parametre oranı: %0.11


In [5]:
import pathlib
import locale
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# 1. Windows Türkçe dil sorununu kökünden çözen yama (Monkey-Patch)
original_read_text = pathlib.Path.read_text
def custom_read_text(self, encoding=None, errors=None):
    return original_read_text(self, encoding="utf-8", errors=errors)
pathlib.Path.read_text = custom_read_text
locale.getpreferredencoding = lambda do_setlocale=True: "utf-8"

# 2. VRAM (Ekran Kartı Hafızası) Taştığı için max_length 512 yapıldı!
print("⏳ Veri seti modele uygun hale getiriliyor (Tokenize işlemi)...")
def tokenize_function(examples):
    # 4 GB VRAM'e sığması için 1024 yerine 512 kullanıyoruz
    return tokenizer(examples["text"], truncation=True, max_length=512)

# Veriyi hızlıca map fonksiyonu ile işliyoruz
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)

# 3. Kaya gibi sağlam, klasik Training ayarları
training_args = TrainingArguments(
    output_dir="./qwen-epikriz-model",
    num_train_epochs=1,              
    per_device_train_batch_size=1,   
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,                       
    logging_steps=10,
    save_steps=50,
)

# 4. Hata veren SFTTrainer yerine, orijinal ve stabil Trainer'ı kullanıyoruz
trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

print("🚀 Her şey hazır! Eğitim nihayet başlıyor...")
trainer.train()
print("✅ Eğitim başarıyla tamamlandı!")

⏳ Veri seti modele uygun hale getiriliyor (Tokenize işlemi)...


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

🚀 Her şey hazır! Eğitim nihayet başlıyor...


C:\Users\ASUS\anaconda3\envs\epikriz\lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.155654
20,1.932736
30,1.796078
40,1.639612
50,1.458822
60,1.447401
70,1.491512
80,1.536849
90,1.463409
100,1.481343


C:\Users\ASUS\anaconda3\envs\epikriz\lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
C:\Users\ASUS\anaconda3\envs\epikriz\lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
C:\Users\ASUS\anaconda3\

✅ Eğitim başarıyla tamamlandı!


In [10]:
import torch

model.config.use_cache = True
model.eval()
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

print("🩺 Doktor Qwen kendine geldi, yeniden test ediliyor...\n")

hasta_sikayeti = "Hasta 22 yaşında erkek. Ellerde istirahat tremoru, konuşmada bozulma (dizartri) ve yutma güçlüğü şikayetleriyle başvurdu. Göz muayenesinde korneada Kayser-Fleischer halkası izlendi. Laboratuvar testlerinde serum seruloplazmin seviyesi çok düşük, 24 saatlik idrar bakır atılımı belirgin şekilde artmış bulundu."
prompt = f"<|im_start|>user\nŞu hasta şikayetine göre profesyonel ve tıbbi terminolojiye uygun bir epikriz raporu yaz: {hasta_sikayeti}<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    cikti = model.generate(
        **inputs,
        max_new_tokens=1024, 
        temperature=0.3,          # 0.7'den 0.3'e düşürdük: Daha ciddi ve mantıklı yazmasını sağlar
        repetition_penalty=1.2,   # İŞTE ÇÖZÜM: Aynı kelimeyi tekrarlamasını kesin olarak yasaklar
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

sonuc_metni = tokenizer.decode(cikti[0], skip_special_tokens=True)

print("-" * 40)
print("📝 ÜRETİLEN EPİKRİZ RAPORU:")
print("-" * 40)
print(sonuc_metni.split("assistant\n")[-1])

🩺 Doktor Qwen kendine geldi, yeniden test ediliyor...

----------------------------------------
📝 ÜRETİLEN EPİKRİZ RAPORU:
----------------------------------------
### Epikris Raporu

**Hastane Bilgileri:**
- **Ad:** [İsim]
- **Yaş:** 22
- **Cinsiyet:** Erkeğin
- **Takip Numarası:** [Numara]

#### Tanım:
Bu raporda, 22 yaşındaki erkeğe ait hastalığın detaylı tanımlanması sunulmaktadır.

#### Anamnez:
Bazı anamnese bilgileri mevcuttur ancak bu konuda daha fazla ayrıntı verilemez.

#### Tıp İlişkisi:
Hasta, ellere istihata tremuru, konuşma bozan dızarı ve yutma güçlükleri ile ilgilenecektir. Ayrıca göz müyensizde kayser-fleischer halkası bulunan bir kişinin bildirildiği görülmüştür.
 
#### Muayeneler:
Göz müyensizde kayser-fleischer halkası izlenmiştir. 

#### Laboratuvar Testleri:
Serum seruloplazmin düzeyinin düşük olduğu ve 24 saatlik idrağı da gösterdiği laboratuvar sonuçları bulunmaktadır.

#### Sonuçlar:
Hasta üzerinde kayser-fleischer halkası bulunmuş olup, bu nedenle endokrin sis